# App-15 : Planification de Calendrier Sportif (CSP)

## Objectifs d'apprentissage

Ce notebook explore la planification de calendriers sportifs comme un **Probleme de Satisfaction de Contraintes (CSP)**. A la fin de ce notebook, vous saurez :

1. **Modeliser** un championnat sportif en CSP (variables, domaines, contraintes)
2. **Implementer** des contraintes hard et soft avec OR-Tools CP-SAT
3. **Gerer** les contraintes TV, de deplacement et d'equitablete
4. **Optimiser** le calendrier selon plusieurs criteres

## Contexte

La planification de calendriers sportifs est un probleme NP-difficile avec des applications dans tous les championnats professionnels (Ligue 1, NBA, NFL). Les contraintes incluent :

- **Format round-robin** : chaque equipe joue contre toutes les autres
- **Contraintes TV** : matchs attractifs a certains creneaux
- **Equitablete geographique** : minimiser les deplacements
- **Contraintes de stade** : disponibilite des enceintes
- **Sequences** : eviter trop de matchs consecutifs domicile/exterieur

In [ ]:
# Imports
from ortools.sat.python import cp_model
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Set
import random
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import defaultdict

print("OR-Tools CP-SAT charge avec succes")

## 1. Modelisation du Probleme

### Definition du CSP

| Element | Description |
|---------|-------------|
| **Variables** | `match[i,j,r]` = 1 si equipe i recoit equipe j a la ronde r |
| **Domaines** | {0, 1} (binaire) |
| **Contraintes** | Voir ci-dessous |

### Contraintes Hard (obligatoires)

1. **Round-robin simple** : chaque pair d'equipes joue exactement une fois
2. **Une equipe par match** : une equipe joue au plus un match par ronde
3. **Symetrie** : si A vs B, alors pas B vs A dans la meme ronde
4. **Capacite stade** : une equipe ne peut pas recevoir deux fois meme ronde

### Contraintes Soft (preferences)

1. **Equilibre D/E** : alternance domicile/exterieur
2. **Minimiser deplacements** : reduire les km parcourus
3. **TV slots** : matchs attractifs le samedi soir
4. **Rivalries** : derbys dans des rondes specifiques

In [ ]:
@dataclass
class Team:
    """Representation d'une equipe de football."""
    id: int
    name: str
    city: str
    latitude: float
    longitude: float
    stadium_capacity: int
    tv_attractiveness: float  # 0-1, influence TV
    
    def __hash__(self):
        return self.id
    
    def __eq__(self, other):
        return self.id == other.id


@dataclass  
class SportsLeague:
    """Ligue sportive avec equipes et contraintes."""
    name: str
    teams: List[Team]
    rounds: int
    
    def __post_init__(self):
        self.n_teams = len(self.teams)
        # En round-robin simple: n_teams - 1 rondes
        if self.rounds is None:
            self.rounds = self.n_teams - 1
    
    def distance(self, team1: Team, team2: Team) -> float:
        """Calcule la distance en km entre deux villes (approximation)."""
        # Formule haversine simplifiee
        lat1, lon1 = team1.latitude, team1.longitude
        lat2, lon2 = team2.latitude, team2.longitude
        
        R = 6371  # Rayon terre en km
        phi1, phi2 = np.radians([lat1, lat2])
        delta_phi = np.radians(lat2 - lat1)
        delta_lambda = np.radians(lon2 - lon1)
        
        a = np.sin(delta_phi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        
        return R * c


def create_ligue1_sample(n_teams: int = 8) -> SportsLeague:
    """Cree un echantillon de la Ligue 1 (simplifie)."""
    # Coordonnees approximatives des villes francaises
    cities = [
        ("Paris", 48.8566, 2.3522),
        ("Marseille", 43.2965, 5.3698),
        ("Lyon", 45.7640, 4.8357),
        ("Lille", 50.6292, 3.0573),
        ("Bordeaux", 44.8378, -0.5792),
        ("Nantes", 47.2184, -1.5536),
        ("Nice", 43.7102, 7.2620),
        ("Toulouse", 43.6047, 1.4442),
        ("Rennes", 48.1173, -1.6778),
        ("Strasbourg", 48.5734, 7.7521),
        ("Montpellier", 43.6119, 3.8772),
        ("Reims", 49.2583, 4.0317),
    ]
    
    teams = []
    for i in range(min(n_teams, len(cities))):
        city_name, lat, lon = cities[i]
        teams.append(Team(
            id=i,
            name=f"{city_name} FC",
            city=city_name,
            latitude=lat,
            longitude=lon,
            stadium_capacity=random.randint(20000, 60000),
            tv_attractiveness=random.uniform(0.3, 1.0)
        ))
    
    return SportsLeague(
        name=f"Ligue 1 (Echantillon {n_teams} equipes)",
        teams=teams,
        rounds=n_teams - 1
    )


# Creation de la ligue
league = create_ligue1_sample(n_teams=6)
print(f"Ligue creee: {league.name}")
print(f"Equipes: {[t.name for t in league.teams]}")
print(f"Rondes: {league.rounds}")

## 2. Calcul de la Matrice de Distances

La matrice de distances est cruciale pour optimiser les deplacements.

In [ ]:
def compute_distance_matrix(league: SportsLeague) -> np.ndarray:
    """Calcule la matrice des distances entre toutes les equipes."""
    n = league.n_teams
    distances = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i+1, n):
            d = league.distance(league.teams[i], league.teams[j])
            distances[i, j] = d
            distances[j, i] = d
    
    return distances


# Calcul et affichage
distances = compute_distance_matrix(league)

plt.figure(figsize=(8, 6))
plt.imshow(distances, cmap='YlOrRd')
plt.colorbar(label='Distance (km)')
plt.xticks(range(league.n_teams), [t.city for t in league.teams], rotation=45)
plt.yticks(range(league.n_teams), [t.city for t in league.teams])
plt.title('Matrice des Distances')
plt.tight_layout()
plt.show()

print("\nDistances (km):")
df_distances = pd.DataFrame(
    distances.astype(int),
    index=[t.city for t in league.teams],
    columns=[t.city for t in league.teams]
)
display(df_distances)

## 3. Modelisation CP-SAT

Nous utilisons OR-Tools CP-SAT pour resoudre le probleme.

In [ ]:
class SportsScheduler:
    """Solveur CP-SAT pour la planification sportive."""
    
    def __init__(self, league: SportsLeague, distances: np.ndarray):
        self.league = league
        self.distances = distances
        self.model = cp_model.CpModel()
        self.solver = cp_model.CpSolver()
        
        # Variables de decision
        self.match_vars = {}  # (home, away, round) -> BoolVar
        self._create_variables()
        self._add_constraints()
        
    def _create_variables(self):
        """Cree les variables de decision."""
        n = self.league.n_teams
        rounds = self.league.rounds
        
        # match[i,j,r] = 1 si equipe i recoit equipe j a la ronde r
        for r in range(rounds):
            for i in range(n):
                for j in range(n):
                    if i != j:
                        self.match_vars[(i, j, r)] = self.model.NewBoolVar(
                            f'match_{i}_{j}_r{r}'
                        )
    
    def _add_constraints(self):
        """Ajoute les contraintes au modele."""
        n = self.league.n_teams
        rounds = self.league.rounds
        
        # C1: Chaque pair d'equipes joue exactement une fois (round-robin simple)
        for i in range(n):
            for j in range(i+1, n):
                # Soit i recoit j, soit j recoit i, exactement une fois
                self.model.Add(
                    sum(self.match_vars[(i, j, r)] + self.match_vars[(j, i, r)]
                        for r in range(rounds)) == 1
                )
        
        # C2: Chaque equipe joue exactement une fois par ronde
        for r in range(rounds):
            for i in range(n):
                # Soit l'equipe i joue a domicile, soit a l'exterieur
                self.model.Add(
                    sum(self.match_vars[(i, j, r)] + self.match_vars[(j, i, r)]
                        for j in range(n) if j != i) == 1
                )
        
        # C3: Au plus un match a domicile par ronde (implicitement satisfait par C2)
        # C4: Au plus un match a l'exterieur par ronde (implicitement satisfait par C2)
    
    def add_balance_constraint(self, max_consecutive: int = 2):
        """
        Ajoute une contrainte d'equilibre domicile/exterieur.
        Evite plus de max_consecutive matchs consecutifs a domicile ou exterieur.
        """
        n = self.league.n_teams
        rounds = self.league.rounds
        
        for i in range(n):
            for r in range(rounds - max_consecutive):
                # Somme des matchs domicile sur max_consecutive+1 rondes
                home_streak = sum(
                    sum(self.match_vars[(i, j, r+k)] for j in range(n) if j != i)
                    for k in range(max_consecutive + 1)
                )
                self.model.Add(home_streak <= max_consecutive)
                
                # Meme chose pour l'exterieur
                away_streak = sum(
                    sum(self.match_vars[(j, i, r+k)] for j in range(n) if j != i)
                    for k in range(max_consecutive + 1)
                )
                self.model.Add(away_streak <= max_consecutive)
    
    def minimize_travel(self, weight: int = 100):
        """
        Ajoute un objectif de minimisation des deplacements.
        """
        n = self.league.n_teams
        rounds = self.league.rounds
        
        travel_terms = []
        for r in range(rounds):
            for i in range(n):
                for j in range(n):
                    if i != j:
                        # Distance parcourue si j joue a l'exterieur chez i
                        travel_terms.append(
                            int(self.distances[i, j]) * self.match_vars[(i, j, r)]
                        )
        
        self.model.Minimize(sum(travel_terms))
    
    def solve(self, time_limit: int = 30) -> bool:
        """Resout le probleme."""
        self.solver.parameters.max_time_in_seconds = time_limit
        self.solver.parameters.num_search_workers = 4
        
        status = self.solver.Solve(self.model)
        return status == cp_model.OPTIMAL or status == cp_model.FEASIBLE
    
    def get_schedule(self) -> Dict[int, List[Tuple[int, int]]]:
        """Retourne le calendrier solution."""
        # Utiliser StatusName() pour verifier le statut (correction API OR-Tools)
        if self.solver.StatusName() == "UNKNOWN":
            return {}
        
        schedule = {r: [] for r in range(self.league.rounds)}
        
        for (i, j, r), var in self.match_vars.items():
            if self.solver.Value(var) == 1:
                schedule[r].append((i, j))  # i recoit j
        
        return schedule


# Creation et resolution du probleme
scheduler = SportsScheduler(league, distances)
scheduler.add_balance_constraint(max_consecutive=2)
scheduler.minimize_travel()

print("Resolution en cours...")
if scheduler.solve(time_limit=10):
    print("Solution trouvee!")
    schedule = scheduler.get_schedule()
else:
    print("Pas de solution trouvee")
    schedule = {}

## 4. Visualisation du Calendrier

In [ ]:
def display_schedule(league: SportsLeague, schedule: Dict[int, List[Tuple[int, int]]]):
    """Affiche le calendrier de maniere lisible."""
    print(f"\n=== CALENDRIER {league.name} ===\n")
    
    for r in sorted(schedule.keys()):
        print(f"Ronde {r+1}:")
        for home, away in schedule[r]:
            home_team = league.teams[home]
            away_team = league.teams[away]
            print(f"  {home_team.name} vs {away_team.name}")
        print()


def compute_schedule_stats(league: SportsLeague, schedule: Dict[int, List[Tuple[int, int]]],
                          distances: np.ndarray) -> Dict:
    """Calcule les statistiques du calendrier."""
    n = league.n_teams
    
    # Matchs domicile/exterieur par equipe
    home_games = defaultdict(int)
    away_games = defaultdict(int)
    total_travel = defaultdict(float)
    
    for r, matches in schedule.items():
        for home, away in matches:
            home_games[home] += 1
            away_games[away] += 1
            total_travel[away] += distances[home, away]
    
    return {
        'home_games': dict(home_games),
        'away_games': dict(away_games),
        'total_travel': dict(total_travel),
        'avg_travel': np.mean(list(total_travel.values())),
        'travel_std': np.std(list(total_travel.values()))
    }


# Affichage
if schedule:
    display_schedule(league, schedule)
    
    stats = compute_schedule_stats(league, schedule, distances)
    print("\n=== STATISTIQUES ===")
    print(f"Distance totale moyenne par equipe: {stats['avg_travel']:.0f} km")
    print(f"Ecart-type des deplacements: {stats['travel_std']:.0f} km")
    
    print("\nDeplacements par equipe:")
    for i, team in enumerate(league.teams):
        print(f"  {team.name}: {stats['total_travel'][i]:.0f} km "
              f"({stats['home_games'][i]} D, {stats['away_games'][i]} E)")

## 5. Contraintes Avancees

### Contraintes TV

Les chaines de television veulent des matchs attractifs a certains creneaux.

In [ ]:
class TVScheduler(SportsScheduler):
    """Extension avec contraintes TV."""
    
    def __init__(self, league: SportsLeague, distances: np.ndarray):
        super().__init__(league, distances)
        self.tv_slots = {}  # ronde -> (slot, importance)
    
    def add_tv_slot(self, round_idx: int, slot_name: str, importance: float):
        """
        Definit un creneau TV important.
        importance: poids pour les matchs attractifs dans ce creneau.
        """
        self.tv_slots[round_idx] = (slot_name, importance)
    
    def maximize_tv_attractiveness(self):
        """
        Optimise l'attractivite TV des matchs dans les creneaux importants.
        """
        n = self.league.n_teams
        
        attractiveness_terms = []
        
        for r, (slot_name, importance) in self.tv_slots.items():
            for i in range(n):
                for j in range(n):
                    if i != j:
                        # Attractivite combinee des deux equipes
                        combined = (
                            self.league.teams[i].tv_attractiveness +
                            self.league.teams[j].tv_attractiveness
                        )
                        # On veut MAXIMISER l'attractivite
                        attractiveness_terms.append(
                            int(importance * combined * 100) * self.match_vars[(i, j, r)]
                        )
        
        # On maximise l'attractivite (donc on minimise le negatif)
        self.model.Minimize(-sum(attractiveness_terms))


# Exemple avec contraintes TV
tv_scheduler = TVScheduler(league, distances)
tv_scheduler.add_balance_constraint(max_consecutive=2)

# Creneaux TV importants: premiere et derniere ronde
tv_scheduler.add_tv_slot(0, "Ouverture", importance=2.0)
tv_scheduler.add_tv_slot(league.rounds - 1, "Cloture", importance=2.0)

tv_scheduler.maximize_tv_attractiveness()

print("Resolution avec contraintes TV...")
if tv_scheduler.solve(time_limit=10):
    print("Solution trouvee!")
    tv_schedule = tv_scheduler.get_schedule()
    display_schedule(league, tv_schedule)

## 6. Contraintes de Derby

Les derbys (matchs entre equipes geographiquement proches) peuvent avoir des contraintes speciales pour la securite.

In [ ]:
def find_derbies(league: SportsLeague, distances: np.ndarray, 
                threshold_km: float = 200) -> List[Tuple[int, int]]:
    """
    Identifie les derbys (equipes proches geographiquement).
    """
    derbies = []
    n = league.n_teams
    
    for i in range(n):
        for j in range(i+1, n):
            if distances[i, j] <= threshold_km:
                derbies.append((i, j))
    
    return derbies


def schedule_derbies_on_weekends(league: SportsLeague, derbies: List[Tuple[int, int]],
                                schedule: Dict[int, List[Tuple[int, int]]]) -> Dict[int, List[Tuple[int, int]]]:
    """
    Reorganise le calendrier pour placer les derbys en week-end (simplifie).
    Dans un vrai systeme, on ajouterait cette contrainte au modele CP-SAT.
    """
    # Identification des derbys dans le calendrier
    derby_rounds = {}
    
    for r, matches in schedule.items():
        for home, away in matches:
            if (home, away) in derbies or (away, home) in derbies:
                derby_rounds[(home, away)] = r
    
    return derby_rounds


# Identification des derbys
derbies = find_derbies(league, distances, threshold_km=300)

print(f"Derbys identifies (distance < 300km):")
for i, j in derbies:
    print(f"  {league.teams[i].name} vs {league.teams[j].name} "
          f"({distances[i, j]:.0f} km)")

if schedule:
    derby_rounds = schedule_derbies_on_weekends(league, derbies, schedule)
    print(f"\nDerbys dans le calendrier:")
    for (home, away), r in derby_rounds.items():
        print(f"  Ronde {r+1}: {league.teams[home].name} vs {league.teams[away].name}")

## 7. Extension: Round-Robin Double

Dans un championnat double, chaque equipe joue deux fois contre chaque autre (une fois a domicile, une fois a l'exterieur).

In [ ]:
class DoubleRoundRobinScheduler(SportsScheduler):
    """
    Solveur pour round-robin double.
    Chaque pair d'equipes joue deux fois (aller-retour).
    """
    
    def _add_constraints(self):
        """Surcharge pour round-robin double."""
        n = self.league.n_teams
        rounds = self.league.rounds
        
        # C1: Chaque pair d'equipes joue exactement deux fois
        # Une fois i recoit j, une fois j recoit i
        for i in range(n):
            for j in range(i+1, n):
                # i recoit j exactement une fois
                self.model.Add(
                    sum(self.match_vars[(i, j, r)] for r in range(rounds)) == 1
                )
                # j recoit i exactement une fois
                self.model.Add(
                    sum(self.match_vars[(j, i, r)] for r in range(rounds)) == 1
                )
        
        # C2: Chaque equipe joue exactement une fois par ronde
        for r in range(rounds):
            for i in range(n):
                self.model.Add(
                    sum(self.match_vars[(i, j, r)] + self.match_vars[(j, i, r)]
                        for j in range(n) if j != i) == 1
                )


# Exemple round-robin double (2*(n-1) rondes)
double_league = create_ligue1_sample(n_teams=4)
double_distances = compute_distance_matrix(double_league)

# Pour double round-robin: 2*(n-1) rondes
double_league.rounds = 2 * (double_league.n_teams - 1)

double_scheduler = DoubleRoundRobinScheduler(double_league, double_distances)
double_scheduler.minimize_travel()

print(f"Double Round-Robin: {double_league.n_teams} equipes, {double_league.rounds} rondes")

if double_scheduler.solve(time_limit=10):
    double_schedule = double_scheduler.get_schedule()
    display_schedule(double_league, double_schedule)

## 8. Visualisation Graphique du Calendrier

In [ ]:
def plot_schedule_matrix(league: SportsLeague, schedule: Dict[int, List[Tuple[int, int]]]):
    """
    Affiche le calendrier comme une matrice equipes x rondes.
    """
    n = league.n_teams
    rounds = league.rounds
    
    # Matrice: -1 = exterieur, 0 = bye, 1 = domicile
    matrix = np.zeros((n, rounds))
    
    for r, matches in schedule.items():
        for home, away in matches:
            matrix[home, r] = 1   # Domicile
            matrix[away, r] = -1  # Exterieur
    
    plt.figure(figsize=(12, 6))
    
    # Couleurs: rouge = domicile, bleu = exterieur
    cmap = plt.cm.RdBu
    
    plt.imshow(matrix, cmap=cmap, aspect='auto', vmin=-1, vmax=1)
    
    plt.yticks(range(n), [t.city for t in league.teams])
    plt.xticks(range(rounds), [f'R{r+1}' for r in range(rounds)])
    
    plt.xlabel('Ronde')
    plt.ylabel('Equipe')
    plt.title(f'Matrice Domicile/Exterieur - {league.name}')
    
    # Legende
    plt.colorbar(label='Domicile (rouge) / Exterieur (bleu)')
    
    plt.tight_layout()
    plt.show()


if schedule:
    plot_schedule_matrix(league, schedule)

## 9. Benchmark: Taille vs Temps de Resolution

In [ ]:
def benchmark_scaling(max_teams: int = 12, time_limit: int = 5):
    """
    Compare le temps de resolution selon le nombre d'equipes.
    """
    import time
    
    results = []
    
    for n in range(4, max_teams + 1, 2):
        league = create_ligue1_sample(n)
        distances = compute_distance_matrix(league)
        
        scheduler = SportsScheduler(league, distances)
        scheduler.minimize_travel()
        
        start = time.time()
        success = scheduler.solve(time_limit=time_limit)
        elapsed = time.time() - start
        
        results.append({
            'n_teams': n,
            'n_rounds': league.rounds,
            'n_vars': len(scheduler.match_vars),
            'solved': success,
            'time': elapsed
        })
        
        print(f"n={n}: {elapsed:.2f}s, {'OK' if success else 'TIMEOUT'}")
    
    return pd.DataFrame(results)


print("Benchmark de scalabilite...")
df_benchmark = benchmark_scaling(max_teams=10)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(df_benchmark['n_teams'], df_benchmark['time'], 'bo-')
plt.xlabel('Nombre d\'equipes')
plt.ylabel('Temps (s)')
plt.title('Temps de resolution')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(df_benchmark['n_teams'], df_benchmark['n_vars'], 'go-')
plt.xlabel('Nombre d\'equipes')
plt.ylabel('Variables')
plt.title('Complexite (variables)')
plt.grid(True)

plt.tight_layout()
plt.show()

## 10. Resume et Comparaison des Approches

### Approches CSP vs Autres

In [ ]:
# Tableau comparatif
comparison_data = {
    'Methode': ['CSP (OR-Tools)', 'Programmation Lineaire', 'Metaheuristiques', 'Generation manuelle'],
    'Optimalite': ['Oui (si temps)', 'Oui (PLNE)', 'Non (approx)', 'Variable'],
    'Scalabilite': ['Moyenne (20-30 equipes)', 'Bonne', 'Excellente', 'Mauvaise'],
    'Flexibilite contraintes': ['Excellente', 'Moyenne', 'Bonne', 'Excellente'],
    'Temps dev': ['Moyen', 'Eleve', 'Moyen', 'Eleve']
}

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

## Conclusion

### Ce que nous avons appris

1. **Modelisation CSP** : Un calendrier sportif peut etre modelise avec des variables binaires representant chaque match

2. **Contraintes** : Les contraintes hard assurent la validite du round-robin, les contraintes soft optimisent la qualite

3. **Trade-offs** : Minimiser les deplacements vs maximiser l'attractivite TV

4. **Scalabilite** : Le probleme croit en O(n^2 * r) variables, mais CP-SAT gere bien jusqu'a 20-30 equipes

### Extensions possibles

- **Contraintes de referees** : Disponibilite des arbitres
- **Contraintes de stade** : Partage de stade (ex: AC Milan / Inter)
- **Contraintes de coupes** : Eviter les conflits avec coupes europeennes
- **Contraintes de repos** : Jours minimums entre matchs

### References

- [OR-Tools CP-SAT](https://developers.google.com/optimization/cp/cp_solver)
- [Sports Scheduling Literature](https://www.sportscheduling.org/)
- [Round Robin Tournament Problem](https://en.wikipedia.org/wiki/Round-robin_tournament)